# Domain_Coverage — which domain is covered by which Findings file?

Run after the three Findings consumers (`Specimen_Findings`, `Measurement_Findings`,
`Instrument_Findings`). Writes `machine_actionable/Domain_Coverage.xlsx`: one row per
domain, with its test-code and DSS counts, the consumer whose scope it falls in, how many
of its DSSs that consumer's file holds, and a coverage status.

The consumer of a domain is computed from the same rules the consumers use
(`SDTM_Domain_Metadata.xlsx`: `Specimen_Based`, `Measurement`, `Instrument_Domain_Rules`,
minus `Consumer_Exclusions`). The result is then checked against what the three output
files actually contain. The run **fails** when:

- a Findings file holds a domain outside its computed scope;
- a covered domain has DSSs in DSS_View that are not in its Findings file;
- a domain with content is missing from `SDTM_Domain_Metadata.xlsx`;
- a domain falls in more than one consumer class.

Status `no consumer class assigned` states a fact, not a decision: no rule places the
domain in a consumer. Only `excluded` rests on a recorded decision (`Consumer_Exclusions`).

## 1. Setup

In [1]:
import pandas as pd
from pathlib import Path
from datetime import datetime
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

BASE_DIR = Path.cwd().parent          # sdtm-findings-graph/
REPO_ROOT = BASE_DIR.parent           # cdisc-for-ai/

DOMAIN_META_FILE = REPO_ROOT / 'sdtm-domain-reference' / 'machine_actionable' / 'SDTM_Domain_Metadata.xlsx'
TEST_IDENTITY_FILE = REPO_ROOT / 'sdtm-test-codes' / 'machine_actionable' / 'SDTM_Test_Identity.xlsx'
INSTR_TEST_FILE = REPO_ROOT / 'sdtm-test-codes' / 'machine_actionable' / 'SDTM_Instrument_Test_Identity.xlsx'
DSS_VIEW_FILE = REPO_ROOT / 'consumer-bases' / 'interim' / 'DSS_View.xlsx'
CONSUMER_FILES = {
    'Specimen_Findings': BASE_DIR / 'machine_actionable' / 'Specimen_Findings.xlsx',
    'Measurement_Findings': BASE_DIR / 'machine_actionable' / 'Measurement_Findings.xlsx',
    'Instrument_Findings': BASE_DIR / 'machine_actionable' / 'Instrument_Findings.xlsx',
}
OUTPUT_FILE = BASE_DIR / 'machine_actionable' / 'Domain_Coverage.xlsx'

for f in [DOMAIN_META_FILE, TEST_IDENTITY_FILE, INSTR_TEST_FILE, DSS_VIEW_FILE, *CONSUMER_FILES.values()]:
    if not f.exists():
        raise FileNotFoundError(f)
    print(f'  {f.relative_to(REPO_ROOT)}')

  sdtm-domain-reference/machine_actionable/SDTM_Domain_Metadata.xlsx
  sdtm-test-codes/machine_actionable/SDTM_Test_Identity.xlsx
  sdtm-test-codes/machine_actionable/SDTM_Instrument_Test_Identity.xlsx
  consumer-bases/interim/DSS_View.xlsx
  sdtm-findings-graph/machine_actionable/Specimen_Findings.xlsx
  sdtm-findings-graph/machine_actionable/Measurement_Findings.xlsx
  sdtm-findings-graph/machine_actionable/Instrument_Findings.xlsx


## 2. Load

In [2]:
meta = pd.read_excel(DOMAIN_META_FILE, sheet_name='Domains', dtype=str).fillna('')
instr_rules = pd.read_excel(DOMAIN_META_FILE, sheet_name='Instrument_Domain_Rules', dtype=str).fillna('')
excl = pd.read_excel(DOMAIN_META_FILE, sheet_name='Consumer_Exclusions', dtype=str).fillna('')
ref_tests = pd.read_excel(TEST_IDENTITY_FILE, sheet_name='Test Codes', dtype=str).fillna('')
instr_tests = pd.read_excel(INSTR_TEST_FILE, sheet_name='Test Codes', dtype=str).fillna('')
view_ms = pd.read_excel(DSS_VIEW_FILE, sheet_name='Measurement_Specs', dtype=str).fillna('')
consumer_ms = {c: pd.read_excel(f, sheet_name='Measurement_Specs', dtype=str).fillna('')
               for c, f in CONSUMER_FILES.items()}
consumer_ti = {c: pd.read_excel(f, sheet_name='Test_Identity', dtype=str).fillna('')
               for c, f in CONSUMER_FILES.items()}

print(f'SDTM_Domain_Metadata domains:     {len(meta)}')
print(f'SDTM_Test_Identity rows:          {len(ref_tests):,}')
print(f'SDTM_Instrument_Test_Identity:    {len(instr_tests):,}')
print(f'DSS_View Measurement_Specs:       {len(view_ms):,}')
for c in CONSUMER_FILES:
    print(f'{c + " Measurement_Specs:":33} {len(consumer_ms[c]):,}')

SDTM_Domain_Metadata domains:     66
SDTM_Test_Identity rows:          5,988
SDTM_Instrument_Test_Identity:    10,386
DSS_View Measurement_Specs:       1,475
Specimen_Findings Measurement_Specs: 169
Measurement_Findings Measurement_Specs: 296
Instrument_Findings Measurement_Specs: 197


## 3. Scope per consumer

The same rules as in the consumer notebooks: class from `SDTM_Domain_Metadata.xlsx`, minus the
domains `Consumer_Exclusions` names for that consumer.

In [3]:
CLASS_RULES = {
    'Specimen_Findings': set(meta.loc[meta['Specimen_Based'] == 'Yes', 'Domain']),
    'Measurement_Findings': set(meta.loc[meta['Measurement'] == 'Yes', 'Domain']),
    'Instrument_Findings': set(instr_rules['Domain']),
}
_classed = [d for c in CLASS_RULES.values() for d in c]
_multi = sorted({d for d in _classed if _classed.count(d) > 1})
if _multi:
    raise RuntimeError(f'Domains in more than one consumer class: {_multi}')

EXCLUDED = {}   # domain -> (consumer, claim ids, reason)
for (consumer, domain), g in excl.groupby(['Consumer', 'Domain']):
    EXCLUDED[domain] = (consumer, ', '.join(g['Claim_ID']), '; '.join(dict.fromkeys(g['Reason'])))

SCOPE = {c: ds - {d for d, (ec, _, _) in EXCLUDED.items() if ec == c} for c, ds in CLASS_RULES.items()}
for c, ds in SCOPE.items():
    print(f'{c:22} {", ".join(sorted(ds))}')

Specimen_Findings      BS, CP, LB, MB, MI, MS, PC, PP
Measurement_Findings   CV, EG, MK, RE, VS
Instrument_Findings    FT, QS, RS


## 4. Check the scope against the output files

Every domain in a Findings file (Measurement_Specs `domain`, Test_Identity `In_Scope_Domains`)
must be in that consumer's computed scope.

In [4]:
problems = []
for c in CONSUMER_FILES:
    in_file = set(consumer_ms[c]['domain'])
    in_file |= {d for v in consumer_ti[c]['In_Scope_Domains'] for d in v.split('; ') if d}
    outside = sorted(in_file - SCOPE[c])
    if outside:
        problems.append(f'{c} holds domains outside its computed scope: {outside}')
if problems:
    raise RuntimeError('\n'.join(problems))
print('Output files agree with the computed scope.')

Output files agree with the computed scope.


## 5. Coverage per domain

In [5]:
testcd_count = {}
for v in ref_tests['SDTM_Domains']:
    for d in v.split('; '):
        if d:
            testcd_count[d] = testcd_count.get(d, 0) + 1
for d in instr_tests['Domain']:
    if d:
        testcd_count[d] = testcd_count.get(d, 0) + 1
dss_count = view_ms['domain'].value_counts().to_dict()

with_content = {d for d, n in testcd_count.items() if n} | {d for d, n in dss_count.items() if n}
missing = sorted(with_content - set(meta['Domain']))
if missing:
    raise RuntimeError(f'Domains with content not in SDTM_Domain_Metadata (run Scope_Check): {missing}')

rows = []
for _, m in meta.iterrows():
    d = m['Domain']
    consumer = next((c for c, ds in SCOPE.items() if d in ds), '')
    n_tc, n_dss = testcd_count.get(d, 0), dss_count.get(d, 0)
    n_in = int((consumer_ms[consumer]['domain'] == d).sum()) if consumer else 0
    reason = ''
    if consumer:
        if n_dss and n_in != n_dss:
            raise RuntimeError(f'{d}: {n_dss} DSSs in DSS_View, {n_in} in {consumer}')
        status = 'covered' if n_dss else 'in scope, no DSS'
    elif d in EXCLUDED:
        ec, claims, why = EXCLUDED[d]
        status = 'excluded'
        reason = f'Excluded from {ec} ({claims}): {why}'
    elif n_tc or n_dss:
        status = 'no consumer class assigned'
    else:
        status = 'no content'
    rows.append({
        'Domain': d, 'Domain_Name': m['Domain_Name'], 'Observation_Class': m['Observation_Class'],
        'Has_Test_Codes': m['Has_Test_Codes'], 'TESTCD_Count': n_tc, 'DSS_Count': n_dss,
        'Consumer': consumer, 'DSS_In_Consumer': n_in, 'Coverage_Status': status,
        'Reason': reason, 'Domain_Notes': m['Notes'],
    })

coverage = pd.DataFrame(rows)
STATUS_ORDER = ['covered', 'in scope, no DSS', 'excluded', 'no consumer class assigned', 'no content']
coverage['_o'] = coverage['Coverage_Status'].map(STATUS_ORDER.index)
coverage = coverage.sort_values(['_o', 'Consumer', 'Domain']).drop(columns='_o').reset_index(drop=True)

summary = coverage.groupby('Coverage_Status', sort=False).agg(
    Domains=('Domain', 'count'), DSS=('DSS_Count', 'sum'), TESTCD=('TESTCD_Count', 'sum'))
print(summary.to_string())
print(f'\nDSS total: {coverage["DSS_Count"].sum():,} (DSS_View: {len(view_ms):,})')
if coverage['DSS_Count'].sum() != len(view_ms):
    raise RuntimeError('DSS counts per domain do not add up to DSS_View')

                            Domains  DSS  TESTCD
Coverage_Status                                 
covered                          10  662   14174
in scope, no DSS                  6    0    1232
excluded                          2  364     120
no consumer class assigned       32  449    1089
no content                       16    0       0

DSS total: 1,475 (DSS_View: 1,475)


## 6. Write

In [6]:
HEADER_FONT = Font(name='Arial', bold=True, size=10, color='FFFFFF')
DATA_FONT = Font(name='Arial', size=10)
WRAP = Alignment(wrap_text=True, vertical='top')
GREEN_HEADER = PatternFill('solid', fgColor='548235')   # SDTM-CT side
YELLOW_HEADER = PatternFill('solid', fgColor='FFD700')  # COSMoS side
GREY_HEADER = PatternFill('solid', fgColor='808080')    # keys, aggregations, status

wb = Workbook()
ws_rm = wb.active
ws_rm.title = 'ReadMe'
readme_font = Font(name='Arial', size=10)
title_font = Font(name='Arial', size=12, bold=True)
section_font = Font(name='Arial', size=10, bold=True)

_summary_lines = [(f'  {s}: {int(r.Domains)} domains, {int(r.DSS):,} DSSs', readme_font)
                  for s, r in summary.iterrows()]

readme_lines = [
    ('Domain_Coverage — which domain is covered by which Findings file', title_font),
    ('', None),
    ('PROVENANCE', section_font),
    (f'Generated: {datetime.now():%Y-%m-%d %H:%M}', readme_font),
    ('Notebook: sdtm-findings-graph/notebooks/Domain_Coverage.ipynb (run after the three Findings notebooks)', readme_font),
    ('Inputs:', readme_font),
    ('  sdtm-domain-reference/machine_actionable/SDTM_Domain_Metadata.xlsx (Domains, Instrument_Domain_Rules, Consumer_Exclusions)', readme_font),
    ('  sdtm-test-codes/machine_actionable/SDTM_Test_Identity.xlsx, SDTM_Instrument_Test_Identity.xlsx (Test Codes)', readme_font),
    ('  consumer-bases/interim/DSS_View.xlsx (Measurement_Specs)', readme_font),
    ('  sdtm-findings-graph/machine_actionable/Specimen_Findings.xlsx, Measurement_Findings.xlsx, Instrument_Findings.xlsx', readme_font),
    ('', None),
    ('PURPOSE', section_font),
    ('One row per domain in SDTM_Domain_Metadata. Says whether the domain is covered by one of the', readme_font),
    ('three Findings files, and if not, why not. A reader can see where content lives before', readme_font),
    ('searching a Findings file for it.', readme_font),
    ('', None),
    ('COLUMNS', section_font),
    ('Domain, Domain_Name, Observation_Class, Has_Test_Codes — SDTM_Domain_Metadata Domains.', readme_font),
    ('  Has_Test_Codes = the domain has a --TESTCD variable; CT need not have a codelist for it (IE, PC).', readme_font),
    ('TESTCD_Count — rows in SDTM_Test_Identity (domain in SDTM_Domains) plus SDTM_Instrument_Test_Identity.', readme_font),
    ('DSS_Count — COSMoS Dataset Specializations in DSS_View for the domain.', readme_font),
    ('Consumer — the Findings file whose scope holds the domain, by the consumers\' own rules', readme_font),
    ('  (Specimen_Based, Measurement, Instrument_Domain_Rules, minus Consumer_Exclusions).', readme_font),
    ('DSS_In_Consumer — DSS rows for the domain in that file\'s Measurement_Specs.', readme_font),
    ('Coverage_Status — see below. Reason — the Consumer_Exclusions claim and reason, for excluded domains.', readme_font),
    ('Domain_Notes — the Notes column of SDTM_Domain_Metadata Domains, as written there.', readme_font),
    ('', None),
    ('COVERAGE STATUS', section_font),
    ('covered — in a consumer\'s scope; all its DSSs are in that file.', readme_font),
    ('in scope, no DSS — in a consumer\'s scope; COSMoS has no DSS for it (Test_Identity only, if any).', readme_font),
    ('excluded — named in Consumer_Exclusions; the one status that rests on a recorded decision.', readme_font),
    ('no consumer class assigned — has test codes or DSSs, but no consumer rule places it. A fact, not a decision.', readme_font),
    ('no content — no test codes and no DSSs.', readme_font),
    ('', None),
    ('SUMMARY', section_font),
    *_summary_lines,
    (f'  DSS total {coverage["DSS_Count"].sum():,} = DSS_View Measurement_Specs.', readme_font),
    ('', None),
    ('CHECKS (the run fails otherwise)', section_font),
    ('Every domain in a Findings file is in that consumer\'s computed scope.', readme_font),
    ('Every covered domain has all its DSS_View DSSs in its Findings file.', readme_font),
    ('Every domain with content is in SDTM_Domain_Metadata; no domain is in two consumer classes.', readme_font),
    ('', None),
    ('HEADER COLOUR CONVENTION', section_font),
    ('Green = SDTM-CT side (TESTCD_Count). Yellow = COSMoS side (DSS_Count). Grey = keys, metadata, status.', readme_font),
    ('', None),
    ('Not an official CDISC product.', readme_font),
]
for ri, (text, font) in enumerate(readme_lines, 1):
    cell = ws_rm.cell(row=ri, column=1, value=text if text else None)
    if font:
        cell.font = font
ws_rm.column_dimensions['A'].width = 110

ws = wb.create_sheet('Domain_Coverage')
FILLS = {c: GREY_HEADER for c in coverage.columns}
FILLS.update({'TESTCD_Count': GREEN_HEADER, 'DSS_Count': YELLOW_HEADER})
WIDTHS = {'Domain': 8, 'Domain_Name': 34, 'Observation_Class': 16, 'Has_Test_Codes': 10,
          'TESTCD_Count': 10, 'DSS_Count': 10, 'Consumer': 22, 'DSS_In_Consumer': 11,
          'Coverage_Status': 26, 'Reason': 60, 'Domain_Notes': 50}
cols = list(coverage.columns)
for ci, name in enumerate(cols, 1):
    cell = ws.cell(row=1, column=ci, value=name)
    cell.font, cell.fill, cell.alignment = HEADER_FONT, FILLS[name], WRAP
for ri, row in enumerate(coverage.itertuples(index=False), 2):
    for ci, val in enumerate(row, 1):
        cell = ws.cell(row=ri, column=ci, value=val if val != '' else None)
        cell.font, cell.alignment = DATA_FONT, WRAP
for ci, name in enumerate(cols, 1):
    ws.column_dimensions[get_column_letter(ci)].width = WIDTHS[name]
ws.freeze_panes = 'A2'
ws.auto_filter.ref = f'A1:{get_column_letter(len(cols))}1'

wb.save(OUTPUT_FILE)
print(f'Written: {OUTPUT_FILE}')
print(f'Domain_Coverage: {len(coverage)} rows x {len(cols)} cols')

Written: /Users/kerstinforsberg/repos/GitHub/cdisc-for-ai/sdtm-findings-graph/machine_actionable/Domain_Coverage.xlsx
Domain_Coverage: 66 rows x 11 cols
